# Link Prediction in Social Networks
## Notebook 04: Friend Recommendation Engine & Triadic Closure Simulation

### Objectives
1. Perform live inference: generate "People You May Know" recommendations for any user in the network.
2. Generate explainable AI breakdowns: explain recommendations using mutual friends and topological similarity.
3. Run a What-If network simulation to empirically validate the **Triadic Closure Principle**.

In [ ]:
import sys
sys.path.append('..')

import networkx as nx
import numpy as np
import pandas as pd

from src.data_loader import load_graph, split_edges_leak_free
from src.pipeline import LinkPredictionPipeline
from src.models import train_model_zoo
from src.recommender import recommend_friends_for_user, analyze_user_pair

### 1. Initialize Pipeline & Model

In [ ]:
G = load_graph(dataset_name="facebook_ego", sample_nodes=800, data_dir="../data/raw")
splits = split_edges_leak_free(G, test_ratio=0.15, val_ratio=0.05, seed=42)
pipeline = LinkPredictionPipeline(use_embeddings=True, embedding_dim=16, seed=42)
data_dict = pipeline.fit_transform_splits(splits)
models = train_model_zoo(data_dict)

best_model = models["XGBoost"]

### 2. Live "People You May Know" Friend Recommendation

In [ ]:
target_user = 0
recommendations = recommend_friends_for_user(target_user, G, pipeline, best_model, top_k=5)

print(f"--- Top Friend Recommendations for User {target_user} ---")
for r in recommendations:
    print(f"Candidate: User {r['candidate_id']:<4} | Probability: {r['probability']*100:.1f}% | AA Score: {r['adamic_adar']:.2f}")
    print(f"   Explanation: {r['explanation']}")
    print("-" * 60)

### 3. Pair Connection Analysis

In [ ]:
u_test, v_test = 5, 25
analysis = analyze_user_pair(u_test, v_test, G, pipeline, best_model)

print(f"Analysis between User {u_test} and User {v_test}:")
print(f"- Already Connected: {analysis['already_connected']}")
print(f"- Predicted Link:     {analysis['predicted_link']}")
print(f"- Link Probability:   {analysis['confidence_score']}%")
print(f"- Mutual Friends:     {analysis['mutual_friends_count']} {analysis['mutual_friends']}")

### 4. Triadic Closure Experiment (What-If Simulation)
Simulate adding mutual friends between two initially unconnected users and observe the probability shift.

In [ ]:
u_sim, v_sim = 10, 50
base = analyze_user_pair(u_sim, v_sim, G, pipeline, best_model)
print(f"Initial Link Probability without shared friends: {base['confidence_score']:.1f}%")

# Add 3 synthetic mutual friends
sim_G = G.copy()
max_n = max(sim_G.nodes())
for i in range(3):
    new_friend = max_n + 100 + i
    sim_G.add_node(new_friend)
    sim_G.add_edge(u_sim, new_friend)
    sim_G.add_edge(v_sim, new_friend)

sim_pipeline = LinkPredictionPipeline(use_embeddings=False)
sim_pipeline.precomputed_structures = pipeline.precomputed_structures
sim_pipeline.scaler = pipeline.scaler
sim_pipeline.feature_names = pipeline.feature_names
sim_pipeline.node2vec_model = pipeline.node2vec_model

after_sim = analyze_user_pair(u_sim, v_sim, sim_G, sim_pipeline, best_model)
print(f"Simulated Link Probability after adding 3 mutual friends: {after_sim['confidence_score']:.1f}%")
print(f"Probability Jump: +{after_sim['confidence_score'] - base['confidence_score']:.1f}%")